# 01. 텍스트 전처리 (Text Preprocessing)

## 학습 목표
- NLP 전처리 파이프라인 전체 흐름 이해
- Tokenization 방식별 차이점 비교
- BoW와 TF-IDF를 직접 구현하고 sklearn과 비교

## 참고 자료
- [Speech and Language Processing - Jurafsky & Martin, Ch.2](https://web.stanford.edu/~jurafsky/slp3/2.pdf)
- [KoNLPy 공식 문서](https://konlpy.org/ko/latest/)

---

In [ ]:
import numpy as np
import re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

## 1. 텍스트 정규화 (Text Normalization)

텍스트 데이터는 **다양한 형태**로 존재한다. 같은 의미라도 표현이 다를 수 있으므로, 통일된 형태로 변환하는 것이 정규화.

| 정규화 단계 | 예시 | ML에서의 역할 |
|------------|------|---------------|
| 소문자화 | "Hello" → "hello" | 어휘 수 감소, 일관성 |
| 특수문자 제거 | "it's" → "its" | 노이즈 제거 |
| 불용어 제거 | "the", "is", "a" 제거 | 의미 없는 단어 제거 |
| 어간 추출 | "running" → "run" | 단어 변형 통합 |

In [ ]:
# 원본 텍스트
text = "Hello, World! This is a Sample Text. NLP is Fun!!! Let's learn NLP."
print(f"원본: {text}")

# 1. 소문자화
text_lower = text.lower()
print(f"소문자화: {text_lower}")

# 2. 특수문자 제거 (알파벳, 숫자, 공백만 남김)
text_clean = re.sub(r'[^a-z0-9\s]', '', text_lower)
print(f"특수문자 제거: {text_clean}")

# 3. 불용어 제거
stop_words = {'a', 'an', 'the', 'is', 'are', 'was', 'this', 'that', 'it', 'lets'}
words = text_clean.split()
words_filtered = [w for w in words if w not in stop_words]
print(f"불용어 제거: {words_filtered}")
print(f"→ 단어 수: {len(words)} → {len(words_filtered)} ({len(words) - len(words_filtered)}개 제거)")

### 정규화 주의사항

무조건 정규화하면 안 되는 경우도 있다:
- **대소문자**: "Apple"(회사) vs "apple"(사과) — 의미가 달라질 수 있음
- **특수문자**: "C++", "$100" — 의미 있는 특수문자
- **불용어**: "to be or not to be" — 불용어가 핵심 의미를 담는 경우

> **실무 팁**: 과제에 따라 정규화 수준을 조절해야 한다. LLM 시대에는 전처리를 최소화하는 추세.

---
## 2. Tokenization

텍스트를 **의미 있는 단위(토큰)**로 분할하는 과정. NLP의 가장 기본적인 전처리.

### 2.1 공백 기반 토큰화

In [ ]:
text = "Natural language processing is a subfield of AI."

# 가장 단순한 방법: 공백으로 분리
tokens_space = text.split()
print(f"공백 분리: {tokens_space}")
print(f"→ 문제: 'AI.'에 마침표가 붙어 있음")

### 2.2 정규식 기반 토큰화

정규식을 사용하면 더 정교한 토큰화가 가능.

In [ ]:
text = "Natural language processing (NLP) is a sub-field of AI. It's amazing!"

# 정규식: 단어 문자(알파벳, 숫자, _)를 기준으로 분리
tokens_word = re.findall(r'\w+', text)
print(f"\\w+ 패턴: {tokens_word}")

# 정규식: 좀 더 정교하게 (하이픈, 축약형 포함)
tokens_refined = re.findall(r"\w+(?:[-']\w+)*", text)
print(f"정교한 패턴: {tokens_refined}")

# 비교
print(f"\n공백 분리: {len(text.split())}개 토큰")
print(f"정규식: {len(tokens_word)}개 토큰")
print(f"정교한 정규식: {len(tokens_refined)}개 토큰")

---
## 3. 한국어 토큰화

### 영어 vs 한국어 토큰화의 차이

| 특성 | 영어 | 한국어 |
|------|------|--------|
| 단어 구분 | 공백으로 대부분 가능 | 공백만으로 불충분 |
| 형태소 | 단순 (접두사/접미사) | 복잡 (조사, 어미 변화) |
| 예시 | "I go to school" | "나는 학교에 간다" |
| 토큰화 결과 | [I, go, to, school] | [나, 는, 학교, 에, 가, ㄴ다] |

한국어는 **교착어**이므로, 조사와 어미가 어근에 결합된다:
- "학교에서" = 학교(명사) + 에서(조사)
- "먹었다" = 먹(어근) + 었(과거시제) + 다(종결어미)

→ **형태소 분석기**가 필요하다.

### KoNLPy 소개

한국어 NLP 패키지로, 여러 형태소 분석기를 통합 제공:

| 분석기 | 특징 |
|--------|------|
| **Okt** (Open Korean Text) | 속도 빠름, 정규화 기능 |
| **Mecab** | 가장 빠름, 설치 복잡 |
| **Kkma** | 정확도 높음, 느림 |
| **Hannanum** | KAIST 개발 |

In [ ]:
# KoNLPy 설치 (Colab에서 실행)
# !pip install konlpy

# KoNLPy 사용 예시 (설치된 환경에서 실행)
# from konlpy.tag import Okt
# okt = Okt()

# text_kr = "자연어 처리는 인공지능의 핵심 분야입니다."
# print(f"형태소 분석: {okt.morphs(text_kr)}")
# print(f"명사 추출: {okt.nouns(text_kr)}")
# print(f"품사 태깅: {okt.pos(text_kr)}")

# KoNLPy 없이도 이해할 수 있도록 결과를 직접 보여줌
text_kr = "자연어 처리는 인공지능의 핵심 분야입니다."
print(f"원문: {text_kr}")
print()

# 공백 기반 (부정확)
tokens_space = text_kr.split()
print(f"공백 분리: {tokens_space}")
print("→ '처리는'에 조사 '는'이 붙어 있어서 '처리'와 다른 토큰이 됨")
print()

# 형태소 분석 결과 (Okt 기준)
morphs_result = ['자연어', '처리', '는', '인공지능', '의', '핵심', '분야', '입니다', '.']
print(f"형태소 분석 (Okt): {morphs_result}")
print("→ '처리는' → '처리' + '는'으로 올바르게 분리")
print()

# 품사 태깅 결과
pos_result = [('자연어', 'Noun'), ('처리', 'Noun'), ('는', 'Josa'),
              ('인공지능', 'Noun'), ('의', 'Josa'), ('핵심', 'Noun'),
              ('분야', 'Noun'), ('입니다', 'Adjective'), ('.', 'Punctuation')]
print(f"품사 태깅: {pos_result}")
print()

# 명사만 추출
nouns = [word for word, pos in pos_result if pos == 'Noun']
print(f"명사 추출: {nouns}")
print("→ 많은 NLP 과제에서 명사만 추출하면 핵심 의미를 파악할 수 있음")

---
## 4. Bag of Words (BoW)

텍스트를 **단어의 출현 빈도 벡터**로 표현하는 방법. 단어 순서는 무시.

예시:
- 문서1: "I like NLP"
- 문서2: "I like deep learning"
- 어휘: {I, like, NLP, deep, learning}

| | I | like | NLP | deep | learning |
|---|---|---|---|---|---|
| 문서1 | 1 | 1 | 1 | 0 | 0 |
| 문서2 | 1 | 1 | 0 | 1 | 1 |

### 4.1 직접 구현

In [ ]:
def build_bow(documents):
    """BoW 벡터를 직접 구현"""
    # Step 1: 어휘 사전 구축
    vocab = set()
    tokenized_docs = []
    for doc in documents:
        tokens = doc.lower().split()
        tokenized_docs.append(tokens)
        vocab.update(tokens)
    
    # 정렬하여 일관된 순서
    vocab = sorted(vocab)
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    
    # Step 2: 각 문서를 벡터로 변환
    bow_matrix = np.zeros((len(documents), len(vocab)))
    for i, tokens in enumerate(tokenized_docs):
        for token in tokens:
            bow_matrix[i, word2idx[token]] += 1
    
    return bow_matrix, vocab

# 테스트
documents = [
    "I like NLP",
    "I like deep learning",
    "NLP uses deep learning"
]

bow_matrix, vocab = build_bow(documents)

print(f"어휘: {vocab}")
print(f"어휘 크기: {len(vocab)}")
print(f"\nBoW 행렬:")
for i, doc in enumerate(documents):
    print(f"  '{doc}' → {bow_matrix[i].astype(int)}")

### 4.2 sklearn CountVectorizer와 비교

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow_sklearn = vectorizer.fit_transform(documents).toarray()

print(f"sklearn 어휘: {vectorizer.get_feature_names_out()}")
print(f"sklearn BoW 행렬:\n{bow_sklearn}")
print(f"\n직접 구현 BoW 행렬:\n{bow_matrix.astype(int)}")
print(f"\n→ 결과가 동일한가? {np.array_equal(bow_sklearn, bow_matrix.astype(int))}")
print("  (어휘 정렬 순서가 같으면 동일)")

### BoW의 한계

1. **단어 순서 무시**: "dog bites man" vs "man bites dog" → 같은 벡터
2. **희소 벡터**: 어휘가 커지면 대부분이 0 → 메모리 비효율
3. **빈도만 반영**: 자주 등장하는 단어가 반드시 중요한 것은 아님 ("the", "is" 등)

---
## 5. TF-IDF

**Term Frequency - Inverse Document Frequency**

BoW의 한계를 보완: 단순 빈도 대신, **특정 문서에서만 자주 나타나는 단어**에 높은 가중치를 부여.

### 수식

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

- **TF (Term Frequency)**: 문서 $d$에서 단어 $t$의 출현 빈도

$$\text{TF}(t, d) = \frac{\text{단어 } t\text{의 문서 } d\text{ 내 출현 횟수}}{\text{문서 } d\text{의 총 단어 수}}$$

- **IDF (Inverse Document Frequency)**: 단어 $t$가 전체 문서에서 얼마나 희귀한지

$$\text{IDF}(t) = \log\frac{N}{1 + \text{df}(t)}$$

  - $N$: 전체 문서 수
  - $\text{df}(t)$: 단어 $t$가 등장하는 문서 수
  - 분모의 +1: 0으로 나누는 것을 방지 (smoothing)

**직관**:
- 모든 문서에 등장하는 "the" → IDF가 낮음 → TF-IDF가 낮음
- 특정 문서에만 등장하는 "transformer" → IDF가 높음 → TF-IDF가 높음

### 5.1 직접 구현

In [ ]:
def compute_tfidf(documents):
    """TF-IDF를 직접 구현"""
    # 토큰화
    tokenized_docs = [doc.lower().split() for doc in documents]
    
    # 어휘 사전
    vocab = sorted(set(w for doc in tokenized_docs for w in doc))
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    
    N = len(documents)
    V = len(vocab)
    
    # TF 계산: 각 문서에서의 정규화된 빈도
    tf_matrix = np.zeros((N, V))
    for i, tokens in enumerate(tokenized_docs):
        counter = Counter(tokens)
        for word, count in counter.items():
            tf_matrix[i, word2idx[word]] = count / len(tokens)
    
    # IDF 계산: log(N / (1 + df))
    idf = np.zeros(V)
    for j, word in enumerate(vocab):
        df = sum(1 for tokens in tokenized_docs if word in tokens)
        idf[j] = np.log(N / (1 + df))
    
    # TF-IDF = TF * IDF
    tfidf_matrix = tf_matrix * idf
    
    return tfidf_matrix, vocab, tf_matrix, idf

# 테스트 문서
documents = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog"
]

tfidf, vocab, tf, idf = compute_tfidf(documents)

print(f"어휘: {vocab}\n")

# IDF 값 확인
print("IDF 값 (값이 클수록 희귀한 단어):")
for word, score in sorted(zip(vocab, idf), key=lambda x: x[1]):
    print(f"  {word:10s}: {score:.4f}")

print(f"\n→ 'the'는 모든 문서에 등장 → IDF가 가장 낮음")
print(f"→ 'mat', 'log', 'chased'는 각각 1개 문서에만 → IDF가 가장 높음")

In [ ]:
# TF-IDF 값 시각화
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(tfidf, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(vocab)))
ax.set_xticklabels(vocab, rotation=45, ha='right')
ax.set_yticks(range(len(documents)))
ax.set_yticklabels([f'Doc{i+1}' for i in range(len(documents))])
ax.set_title('TF-IDF Matrix (값이 클수록 해당 문서에서 중요한 단어)')

# 셀에 값 표시
for i in range(len(documents)):
    for j in range(len(vocab)):
        ax.text(j, i, f'{tfidf[i, j]:.2f}', ha='center', va='center', fontsize=8)

plt.colorbar(im)
plt.tight_layout()
plt.show()

### 5.2 sklearn TfidfVectorizer와 비교

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# sklearn의 TF-IDF
vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(documents).toarray()

print(f"sklearn 어휘: {vectorizer.get_feature_names_out()}")
print(f"\nsklearn TF-IDF:\n{np.round(tfidf_sklearn, 4)}")
print(f"\n직접 구현 TF-IDF:\n{np.round(tfidf, 4)}")

print(f"\n※ sklearn은 IDF 수식이 약간 다름 (smooth_idf=True, L2 정규화 적용)")
print(f"  sklearn IDF: log((1+N)/(1+df)) + 1, 그리고 벡터를 L2 정규화")
print(f"  직접 구현: log(N/(1+df)), 정규화 없음")
print(f"  → 스케일은 다르지만 상대적 중요도 순서는 동일")

---
## 6. 전처리 파이프라인 구축

텍스트를 입력받아 벡터로 변환하는 전체 파이프라인:

```
원본 텍스트 → 정규화 → 토큰화 → 벡터화 (BoW 또는 TF-IDF)
```

In [ ]:
class TextPreprocessor:
    """텍스트 전처리 파이프라인"""
    
    def __init__(self, stop_words=None, method='tfidf'):
        self.stop_words = stop_words or set()
        self.method = method
        self.vocab = None
        self.word2idx = None
        self.idf = None
    
    def normalize(self, text):
        """텍스트 정규화"""
        text = text.lower()
        text = re.sub(r'[^a-z0-9가-힣\s]', '', text)  # 한국어도 허용
        return text
    
    def tokenize(self, text):
        """토큰화 (공백 기반)"""
        tokens = text.split()
        tokens = [t for t in tokens if t not in self.stop_words]
        return tokens
    
    def fit(self, documents):
        """어휘 사전 구축 및 IDF 계산"""
        tokenized_docs = [self.tokenize(self.normalize(doc)) for doc in documents]
        
        # 어휘 사전
        self.vocab = sorted(set(w for doc in tokenized_docs for w in doc))
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}
        
        # IDF 계산 (TF-IDF용)
        N = len(documents)
        self.idf = np.zeros(len(self.vocab))
        for j, word in enumerate(self.vocab):
            df = sum(1 for doc in tokenized_docs if word in doc)
            self.idf[j] = np.log(N / (1 + df))
        
        return self
    
    def transform(self, documents):
        """문서를 벡터로 변환"""
        tokenized_docs = [self.tokenize(self.normalize(doc)) for doc in documents]
        V = len(self.vocab)
        matrix = np.zeros((len(documents), V))
        
        for i, tokens in enumerate(tokenized_docs):
            counter = Counter(tokens)
            for word, count in counter.items():
                if word in self.word2idx:
                    if self.method == 'bow':
                        matrix[i, self.word2idx[word]] = count
                    elif self.method == 'tfidf':
                        tf = count / len(tokens)
                        matrix[i, self.word2idx[word]] = tf * self.idf[self.word2idx[word]]
        
        return matrix
    
    def fit_transform(self, documents):
        return self.fit(documents).transform(documents)


# 파이프라인 테스트
documents = [
    "Machine learning is a branch of artificial intelligence.",
    "Deep learning is a subset of machine learning.",
    "Natural language processing uses machine learning.",
    "Computer vision is another AI application."
]

stop_words = {'a', 'an', 'the', 'is', 'of', 'and', 'in', 'to', 'for'}

# BoW 파이프라인
bow_pipe = TextPreprocessor(stop_words=stop_words, method='bow')
bow_result = bow_pipe.fit_transform(documents)
print("=== BoW 결과 ===")
print(f"어휘: {bow_pipe.vocab}")
print(f"행렬 shape: {bow_result.shape}")
print(f"행렬:\n{bow_result.astype(int)}\n")

# TF-IDF 파이프라인
tfidf_pipe = TextPreprocessor(stop_words=stop_words, method='tfidf')
tfidf_result = tfidf_pipe.fit_transform(documents)
print("=== TF-IDF 결과 ===")
print(f"행렬:\n{np.round(tfidf_result, 3)}")

In [ ]:
# 파이프라인 결과를 활용: 문서 간 유사도 계산
from numpy.linalg import norm

def cosine_similarity(a, b):
    if norm(a) == 0 or norm(b) == 0:
        return 0.0
    return np.dot(a, b) / (norm(a) * norm(b))

print("=== TF-IDF 기반 문서 유사도 ===")
n = len(documents)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(tfidf_result[i], tfidf_result[j])

# 유사도 히트맵
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
labels = [f'Doc{i+1}' for i in range(n)]
ax.set_xticks(range(n))
ax.set_xticklabels(labels)
ax.set_yticks(range(n))
ax.set_yticklabels(labels)
ax.set_title('Document Similarity (TF-IDF + Cosine)')

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center')

plt.colorbar(im)
plt.tight_layout()
plt.show()

# 가장 유사한 문서 쌍 찾기
max_sim = 0
max_pair = (0, 0)
for i in range(n):
    for j in range(i+1, n):
        if sim_matrix[i, j] > max_sim:
            max_sim = sim_matrix[i, j]
            max_pair = (i, j)

print(f"\n가장 유사한 쌍: Doc{max_pair[0]+1} & Doc{max_pair[1]+1} (유사도: {max_sim:.4f})")
print(f"  Doc{max_pair[0]+1}: {documents[max_pair[0]]}")
print(f"  Doc{max_pair[1]+1}: {documents[max_pair[1]]}")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 한국어 문서 TF-IDF 분석

아래 한국어 문서들에 대해 TF-IDF를 계산하고, 각 문서의 핵심 키워드(TF-IDF가 가장 높은 단어)를 추출하세요.

In [ ]:
# 한국어 문서 (형태소 분석 완료된 상태라고 가정 — 공백 기반 토큰화 가능)
kr_documents = [
    "딥러닝 모델 학습 데이터 전처리 방법",
    "자연어 처리 텍스트 전처리 토큰화",
    "딥러닝 자연어 처리 모델 트랜스포머",
    "데이터 분석 시각화 방법 도구"
]

# TODO: TF-IDF를 계산하고 각 문서의 핵심 키워드(상위 2개)를 출력하세요
# 힌트: compute_tfidf 함수를 활용하거나, TextPreprocessor를 수정하세요


### 연습 2: 전처리 파이프라인 확장

TextPreprocessor 클래스에 **어간 추출(stemming)** 기능을 추가하세요.

In [ ]:
# TODO: TextPreprocessor를 확장하여 간단한 영어 어간 추출 기능을 추가하세요
# 규칙 예시:
#   - "ing" 제거: running → runn (간단 버전)
#   - "ed" 제거: played → play
#   - "s" 제거: cats → cat
#
# 테스트 문서:
test_docs = [
    "The cats are running in the park",
    "A cat played with dogs",
    "Dogs are playing and running"
]

# 어간 추출 적용 전후의 BoW 결과를 비교하세요
# 기대 효과: 'cat'과 'cats', 'running'과 'running' 등이 같은 토큰으로 통합


---
## 핵심 정리

| 개념 | 설명 | ML에서의 역할 |
|------|------|---------------|
| 텍스트 정규화 | 소문자화, 특수문자 제거, 불용어 제거 | 노이즈 제거, 어휘 축소 |
| 토큰화 | 텍스트를 의미 단위로 분할 | NLP의 첫 단계, 모든 후속 처리의 기반 |
| 한국어 형태소 분석 | 조사/어미를 분리하는 토큰화 | KoNLPy (Okt, Mecab) 사용 |
| Bag of Words | 단어 출현 빈도 벡터 | 단순하지만 효과적, 기본 문서 표현 |
| TF-IDF | 빈도 + 희소성 가중치 | 문서 핵심 키워드 추출, 검색 엔진 |
| 전처리 파이프라인 | 정규화 → 토큰화 → 벡터화 | 재현 가능한 일관된 처리 |

**다음 노트북**: [02-word-embeddings.ipynb](02-word-embeddings.ipynb) - 단어 임베딩 (Word2Vec, GloVe)